In [ ]:
from datetime import datetime

import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import mean_absolute_error


1.データの読み込み

In [ ]:
df_raw = pd.read_csv("../data/processed/jma_tepco_merged/raw.csv", encoding="utf-8")

In [ ]:
# データの中身の確認
df_raw.head(5)

In [ ]:
# 前処理用にコピー
df_processed = df_raw.copy()

2.前処理

2.1 欠損値の処理

In [ ]:
# 欠損値が多いカラムを削除
list_remove_col_name =["cloud_cover", "solar_radiation", "snow_depth", "snowfall"]

for col in list_remove_col_name:
    df_processed = df_processed.drop(df_processed.filter(like=col).columns, axis=1)

In [ ]:
# カラム削除の確認
df_processed.head(5)

In [ ]:
# 欠損値をうめる
# 時系列データは、前後の値の平均で補完する

list_interpolate_features = ["temperature", "precipitation", "sunshine_duration", "wind_speed", 
                             "dew_point", "vapor_pressure", "humidity", "local_pressure", "sea_level_pressure"]


for feature_name in list_interpolate_features:
    for col_name in df_processed.filter(like=feature_name).columns:
        
        if "_flag" not in col_name:
            # print("col_name: ", col_name)
            df_processed[col_name] = df_processed[col_name].astype(float).interpolate(method="linear")
        # else:
        #     print("flag cal")


In [ ]:
#　欠損値うめる
# 欠損値の多い、tokyo_weather は東京都地理的に近い、神奈川、千葉、埼玉の３つの都道府県で一番多いものを入れる。
#　3県でバラバラの時は、一旦神奈川の天気を入れる

df_processed["tokyo_weather"] = df_processed[["kanagawa_weather", "saitama_weather","chiba_weather"]].mode(axis=1)[0]

2.2 質的変数の型をカテゴリに変換

In [ ]:
# lightgbmで文字型を直接扱えるので、ラベルエンコーディングは飛ばす

list_categorical_cols = ["weather", "wind_direction"]

for feature_name in list_categorical_cols:
    for col_name in df_processed.filter(like=feature_name).columns:
        
        df_processed[col_name] = df_processed[col_name].astype("category")
        # print(col_name)

In [ ]:
# ここまでの前処理は精度改善フェーズと共通とする。
# そのため、ファイルの保存しておく
df_processed.to_csv("../data/processed/dataset/baseline.csv", index=False)

2.3 時間の特徴量の作成

In [ ]:
df_processed["datetime"] = pd.to_datetime(df_processed["datetime"])

In [ ]:
df_processed["month"] = df_processed["datetime"].dt.month.astype(int)

In [ ]:
df_processed["hour"] = df_processed["datetime"].dt.hour.astype(int)


3.ベースラインモデルの構築

3.1 データセットの作成

In [ ]:
# 学習データの期間: 2021/01/01 00:00 ~ 2023/12/31 23:00
# 検証データの期間: 2024/01/01 00:00 ~ 2024/12/31 23:00
# テストデータの期間: 2025/01/01 00:00 ~ 2025/12/31 23:00

val_start =  datetime(2024, 1, 1, 0, 0, 0)
test_start =  datetime(2025, 1, 1, 0, 0, 0)

df_train = df_processed[df_processed["datetime"] < val_start].copy()
df_val = df_processed[(df_processed["datetime"] >= val_start) & (df_processed["datetime"] < test_start)].copy()
df_test = df_processed[df_processed["datetime"] >= test_start].copy()


In [ ]:
# 期間があってるか確認
df_train.head(2)
df_train.tail(2)


In [ ]:
# 期間があってるか確認
df_val.head(2)
df_val.tail(2)


In [ ]:
# 期間があってるか確認
df_test.head(2)
df_test.tail(2)


In [ ]:
# baseline_cols = ["month", "hour"]
# baseline_cols.extend(df_processed.filter(like="temperature").columns)

# print(baseline_cols)


In [ ]:
# データセット作成

y_col_name = "demand"
drop_cols = [y_col_name, "datetime"]

# baselineは、電力需要変動に影響が大きそうな時間、気温、のみで試す
baseline_cols = ["month", "hour"]
baseline_cols.extend(df_processed.filter(like="temperature").columns)

X_train = df_train[baseline_cols]
# X_train = df_train.drop(drop_cols, axis=1)
y_train =df_train[y_col_name]
y_train.name = y_col_name

X_val = df_val[baseline_cols]
y_val = df_val[y_col_name]
y_val.name = y_col_name

X_test = df_test[baseline_cols]
y_test = df_test[y_col_name]
y_test.name = y_col_name

In [ ]:
X_test.columns

3.2 モデルの学習

In [ ]:
model = lgb.LGBMRegressor(
        random_state = 42,
    )

In [ ]:
eval_set = [(X_val.reset_index(drop=True), y_val.reset_index(drop=True))]

model.fit(
    X_train, 
    y_train,
    eval_set = eval_set,
    # verbose=-1 # 学習ログを省略
)

3.3 モデルの予測と精度

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
print("MAE:", mae)